# Unsupervised — seleção completa e cache-aware

Este notebook executa a pipeline normal. Quando os experimentos já foram processados no cluster, todas as etapas encontram o cache e não treinam novamente. Se um artefato estiver ausente, somente a etapa faltante é calculada.

Fluxo: `treino → validação interna escolhe dimensão+parâmetros → 5 finalistas → teste interno → dataset_validation escolhe o embedding → campeão geral`.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay

here = Path.cwd().resolve()
BACKEND = here if (here / 'machine_learning').exists() else (here / 'backend' if (here / 'backend').exists() else here.parent)
sys.path.insert(0, str(BACKEND))
from machine_learning.cache import ModelCache
from machine_learning.unsupervised.runner import (
    UNSUPERVISED_MODELS, finalize_anomaly_model, build_unsupervised_leaderboard
)

## 1. Executar ou recuperar a pipeline

In [ ]:
# False: usa tudo que já existe e calcula apenas o que estiver faltando.
# True: ignora os caches e refaz os experimentos (use preferencialmente no SLURM).
FORCE_RETRAIN = False
summaries = {}
for model in UNSUPERVISED_MODELS:
    summaries[model] = finalize_anomaly_model(model, force_retrain=FORCE_RETRAIN)
pd.DataFrame([{
    'modelo': model, 'protocolo': data['protocol_version'],
    'grids': len(data['finalists']), 'seeds_por_finalista': len(data['seeds'])
} for model, data in summaries.items()])

## 2–4. Espaço de busca, grid por embedding e vencedor interno

In [ ]:
for model, summary in summaries.items():
    print(f'\n### {model.upper()}')
    for finalist in summary['finalists']:
        grid = pd.DataFrame(finalist['grid'])
        display(pd.DataFrame([{
            'embedding': finalist['embedding'], 'pca_dim vencedora': finalist['pca_dim'],
            'parâmetros vencedores': finalist['params'],
            'internal_val_f1_macro': finalist['internal_val_f1_macro'],
            'combinações avaliadas': len(grid),
        }]))
        display(grid.sort_values('internal_val_f1_macro', ascending=False).head(20))
        pivot = grid.pivot_table(index='pca_dim', values='internal_val_f1_macro', aggfunc='max')
        display(pivot.T.style.background_gradient(cmap='Blues', axis=1).format('{:.4f}'))

## 5–8. Os mesmos cinco finalistas no teste interno e no dataset_validation

In [ ]:
for model, summary in summaries.items():
    frame = pd.DataFrame(summary['finalists'])
    cols = ['embedding', 'pca_dim', 'test_f1_macro_mean', 'test_f1_macro_std',
            'val_f1_macro_mean', 'val_f1_macro_std', 'val_pr_auc_mean', 'val_recall_scam_mean']
    display(frame[cols].style.format({c: '{:.4f}' for c in cols[2:]}).highlight_max(subset=['val_f1_macro_mean']))
    plot = frame.set_index('embedding')
    ax = plot[['test_f1_macro_mean', 'val_f1_macro_mean']].plot.bar(
        yerr=plot[['test_f1_macro_std', 'val_f1_macro_std']].to_numpy().T, figsize=(9, 4),
        title=f'{model.upper()}: finalistas por embedding')
    ax.set_ylabel('F1 macro'); ax.set_ylim(0, 1); plt.tight_layout(); plt.show()

## 7. Matrizes de confusão, ROC e PR dos cinco finalistas (seed fixa 42)

In [ ]:
for model, summary in summaries.items():
    fig, axes = plt.subplots(3, 5, figsize=(22, 12))
    for col, finalist in enumerate(summary['finalists']):
        seed42 = next(item for item in finalist['seeds'] if item['seed'] == 42)
        bundle = ModelCache.load_prediction_bundle('unsupervised', seed42['run_id'], 'validation_external')
        y, pred, score = bundle['y_true'], bundle['y_pred'], bundle['y_prob']
        ConfusionMatrixDisplay.from_predictions(y, pred, ax=axes[0, col], colorbar=False)
        RocCurveDisplay.from_predictions(y, score, ax=axes[1, col])
        PrecisionRecallDisplay.from_predictions(y, score, ax=axes[2, col])
        axes[0, col].set_title(finalist['embedding'])
    fig.suptitle(f'{model.upper()} — dataset_validation, seed 42', y=1.01)
    plt.tight_layout(); plt.show()

## 9–10. Ranking dos campeões e campeão geral unsupervised

In [ ]:
leaderboard = build_unsupervised_leaderboard()
display(leaderboard.style.format({c: '{:.4f}' for c in leaderboard.columns if 'mean' in c or 'std' in c}).highlight_max(subset=['val_f1_macro_mean']))
champion = leaderboard.iloc[0]
print(f"Campeão geral: {champion.model_type.upper()} + {champion.embedding}, PCA={champion.pca_dim}, val_f1_macro={champion.val_f1_macro_mean:.4f}")

## Estudo cache-first de tamanho do dataset — top 3

Esta seção apenas lê os artefatos gerados por `submit_cached_model_studies.sh`; não executa tuning ou treinamento.

In [ ]:
study_path = BACKEND / 'experiment_results' / 'unsupervised' / 'studies' / 'dataset_size' / 'aggregate.csv'
if study_path.exists():
    size = pd.read_csv(study_path)
else:
    size = pd.DataFrame()
    print(f'Artefato ausente: {study_path}')
    print('Execute: bash backend/slurm/studies/submit_cached_model_studies.sh --only unsupervised-size')

In [ ]:
if not size.empty:
    fig, axes = plt.subplots(2, 3, figsize=(19, 10))
    for ax, metric in zip(axes.flat, ['f1_macro', 'recall_scam', 'precision_scam', 'pr_auc', 'FP', 'FN']):
        for (candidate, split), group in size.groupby(['candidate', 'split']):
            group = group.sort_values('fraction')
            x, y = group.fraction * 100, group[f'{metric}_mean']
            ci = group.get(f'{metric}_ci95', pd.Series(0, index=group.index))
            ax.plot(x, y, marker='o', label=f'{candidate} — {split}')
            ax.fill_between(x, y-ci, y+ci, alpha=.12)
        ax.set(title=metric, xlabel='% do Ham de ajuste', ylabel=metric); ax.grid(alpha=.25)
    axes[0, -1].legend(fontsize=7, bbox_to_anchor=(1.04, 1), loc='upper left')
    plt.tight_layout(); plt.show()

In [ ]:
if not size.empty:
    test = size[size.split == 'test_internal']
    full = test[test.fraction == 1.0][['candidate', 'f1_macro_mean']].rename(columns={'f1_macro_mean': 'full_f1'})
    saturation = test.merge(full, on='candidate')
    saturation = saturation[saturation.f1_macro_mean >= .95 * saturation.full_f1]
    saturation = saturation.sort_values('fraction').groupby('candidate', as_index=False).first()
    display(saturation[['candidate', 'fraction', 'n_samples', 'f1_macro_mean', 'full_f1']])
    ranks = test.assign(rank=test.groupby('fraction').f1_macro_mean.rank(method='min', ascending=False))
    display(ranks.pivot(index='fraction', columns='candidate', values='rank'))